# BasementDrugDiscovery
## Notebook 07 -- Flexible Side Chain Docking of Top Hits

**What this notebook does:**
Takes the top 5 hits from the rigid docking screen and redocks them against all three CYP51 targets
with binding site side chains set as flexible. Uses GNINA with --flexdist_ligand to automatically
identify flexible residues from the initial rigid docking pose, then redocks at high exhaustiveness.
Results are stored in the same SQLite database with a flexible_docking tag for comparison.

**Why flexible docking:**
The rigid docking screen treats every receptor atom as fixed. Real proteins breathe -- binding site
side chains rotate to accommodate ligands. For 5 compounds this is computationally feasible and
produces more reliable binding poses and scores.

**Tools used:**
- GNINA v1.1 with GPU acceleration (--flexdist_ligand for automatic flexible residue selection)
- prepare_receptor4 from AutoDock Tools (adt_py3 environment) for receptor preparation
- Same SQLite database as Notebooks 05 and 06

**Input:**
- Top 5 hit PDBQT pose files from rigid docking
- Receptor PDB files from Notebook 03
- bdd_molecules.db from Notebook 02

**Output:**
- Flexible docking results in flexible_docking_results table
- Flexible pose PDBQT files
- Comparison table: rigid vs flexible scores

---

### Cell 1 -- Load all required tools

In [ ]:
import os
import json
import sqlite3
import subprocess
import shutil
import tkinter as tk
from tkinter import filedialog, simpledialog
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import ipywidgets as widgets
from IPython.display import display, clear_output

# GNINA binary path
GNINA_BIN = str(Path.home() / 'gnina')

# AutoDock Tools environment
ADT_ENV = str(Path.home() / 'software/anaconda3/envs/adt_py3')
PREPARE_RECEPTOR = str(Path(ADT_ENV) / 'bin/prepare_receptor4')
PREPARE_FLEXRECEPTOR = str(Path(ADT_ENV) / 'bin/prepare_flexreceptor4')

def get_tk_root():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    return root

def browse_file(title='Select file', filetypes=None):
    if filetypes is None:
        filetypes = [('All files', '*.*')]
    root = get_tk_root()
    path = filedialog.askopenfilename(title=title, filetypes=filetypes)
    root.destroy()
    return path

def browse_directory(title='Select folder'):
    root = get_tk_root()
    path = filedialog.askdirectory(title=title)
    root.destroy()
    return path

def ask_string(title, prompt, initial=''):
    root = get_tk_root()
    result = simpledialog.askstring(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

def ask_float(title, prompt, initial=0.0):
    root = get_tk_root()
    result = simpledialog.askfloat(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

def ask_integer(title, prompt, initial=1):
    root = get_tk_root()
    result = simpledialog.askinteger(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

# Verify GNINA
result = subprocess.run([GNINA_BIN, '--version'], capture_output=True, text=True)
print(f'GNINA: {result.stdout.strip()}')

# Verify AutoDock Tools
if Path(PREPARE_RECEPTOR).exists():
    print(f'prepare_receptor4: found at {PREPARE_RECEPTOR}')
else:
    print(f'prepare_receptor4: NOT FOUND at {PREPARE_RECEPTOR}')

print('All tools loaded.')

### Cell 2 -- Select database and output folder

In [ ]:
selected = {
    'db_path': None,
    'output_dir': None
}

status_out = widgets.Output()

btn_db = widgets.Button(description='Browse -- Select bdd_molecules.db',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_output = widgets.Button(description='Browse -- Select output folder',
    button_style='primary', layout=widgets.Layout(width='380px'))

def sel_db(b):
    with status_out:
        p = browse_file('Select bdd_molecules.db',
            [('SQLite DB', '*.db'), ('All files', '*.*')])
        if p:
            selected['db_path'] = p
            print(f'Database: {p}')

def sel_output(b):
    with status_out:
        p = browse_directory('Select output folder for flexible docking')
        if p:
            selected['output_dir'] = p
            Path(p).mkdir(parents=True, exist_ok=True)
            print(f'Output folder: {p}')

btn_db.on_click(sel_db)
btn_output.on_click(sel_output)
display(btn_db)
display(btn_output)
display(status_out)

### Cell 3 -- Initialize flexible docking results table

Adds a new table to the existing database for flexible docking results.
Keeps rigid and flexible results separate for direct comparison.

In [ ]:
def initialize_flexible_table(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS flexible_docking_results (
            result_id           INTEGER PRIMARY KEY AUTOINCREMENT,
            chembl_id           TEXT,
            target_name         TEXT,
            pdb_id              TEXT,
            rigid_affinity      REAL,
            flexible_affinity   REAL,
            affinity_change     REAL,
            flexdist_used       REAL,
            n_flexible_residues INTEGER,
            flexible_residues   TEXT,
            exhaustiveness      INTEGER,
            pose_file           TEXT,
            status              TEXT DEFAULT 'success',
            error_message       TEXT,
            timestamp           TEXT,
            UNIQUE(chembl_id, target_name)
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS flexible_docking_contacts (
            contact_id          INTEGER PRIMARY KEY AUTOINCREMENT,
            result_id           INTEGER,
            chembl_id           TEXT,
            target_name         TEXT,
            residue_name        TEXT,
            residue_number      INTEGER,
            chain               TEXT,
            contact_distance    REAL,
            is_flexible         INTEGER DEFAULT 0,
            FOREIGN KEY (result_id) REFERENCES flexible_docking_results(result_id)
        )
    ''')

    conn.commit()

    existing = cursor.execute(
        'SELECT COUNT(*) FROM flexible_docking_results'
    ).fetchone()[0]

    conn.close()
    print(f'Flexible docking tables initialized.')
    print(f'Existing flexible docking results: {existing}')


if selected['db_path']:
    initialize_flexible_table(selected['db_path'])
else:
    print('Select database in Cell 2 first.')

### Cell 4 -- Select top 5 hits and show rigid docking comparison

Queries the database for the top hits across both fungal targets with selectivity
over human CYP51. These are the compounds we will redock with flexible side chains.

In [ ]:
# Default top 5 based on our analysis
DEFAULT_TOP5 = [
    'CHEMBL3991065',  # Atogepant
    'CHEMBL941',      # Imatinib
    'CHEMBL1171837',  # Ponatinib
    'CHEMBL2105737',  # Sonidegib
    'CHEMBL481',      # Irinotecan
]

top5_out = widgets.Output()
btn_show = widgets.Button(
    description='Show top 5 hits and rigid docking scores',
    button_style='info',
    layout=widgets.Layout(width='400px')
)
btn_custom = widgets.Button(
    description='Enter custom ChEMBL IDs',
    button_style='warning',
    layout=widgets.Layout(width='300px')
)

selected_hits = list(DEFAULT_TOP5)

def show_top5(b):
    with top5_out:
        clear_output()
        if not selected['db_path']:
            print('Select database first.')
            return

        conn = sqlite3.connect(selected['db_path'])

        ids_str = ', '.join(f"'{x}'" for x in selected_hits)
        df = pd.read_sql(f"""
            SELECT
                m.chembl_id, m.name, m.molecular_weight, m.alogp,
                c.affinity_best as candida,
                a.affinity_best as aspergillus,
                h.affinity_best as human,
                c.pose_file as candida_pose,
                a.pose_file as aspergillus_pose,
                h.pose_file as human_pose
            FROM molecules m
            LEFT JOIN docking_results c ON m.chembl_id = c.chembl_id
                AND c.target_name = 'candida_albicanscyp51'
            LEFT JOIN docking_results a ON m.chembl_id = a.chembl_id
                AND a.target_name = 'aspergilluscyp51'
            LEFT JOIN docking_results h ON m.chembl_id = h.chembl_id
                AND h.target_name = 'homo_sapienscyp51'
            WHERE m.chembl_id IN ({ids_str})
        """, conn)

        conn.close()

        print('Top 5 hits -- rigid docking scores across all three targets:')
        print()
        display(df[['chembl_id', 'name', 'molecular_weight', 'alogp',
                     'candida', 'aspergillus', 'human']].to_string(index=False))

        # Store pose files for use in flexible docking
        selected['hit_data'] = df.to_dict('records')
        print(f'\n{len(df)} compounds ready for flexible docking.')

def enter_custom(b):
    with top5_out:
        ids = ask_string(
            'Custom ChEMBL IDs',
            'Enter ChEMBL IDs separated by commas:',
            ','.join(DEFAULT_TOP5)
        )
        if ids:
            selected_hits.clear()
            selected_hits.extend([x.strip() for x in ids.split(',')])
            print(f'Updated hit list: {selected_hits}')

btn_show.on_click(show_top5)
btn_custom.on_click(enter_custom)

print('Click to show the top 5 hits selected for flexible docking:')
display(btn_show)
display(btn_custom)
display(top5_out)

### Cell 5 -- Configure flexible docking targets

For each target specify the receptor PDB file, grid box coordinates,
and the flexibility distance. GNINA will automatically set all receptor
side chains within flexdist Angstroms of the initial docking pose as flexible.

In [ ]:
# Target configuration -- populated via popups
flex_targets = []

target_config_out = widgets.Output()
btn_add_target = widgets.Button(
    description='Add a target for flexible docking',
    button_style='success',
    layout=widgets.Layout(width='350px')
)
btn_show_targets = widgets.Button(
    description='Show configured targets',
    button_style='info',
    layout=widgets.Layout(width='280px')
)

def add_target(b):
    with target_config_out:
        target_name = ask_string('Target Name',
            'Enter target name (e.g. candida_albicanscyp51):', 'candida_albicanscyp51')
        pdb_id = ask_string('PDB ID', 'Enter PDB ID:', '5V5Z')

        receptor_pdbqt = browse_file('Select receptor PDBQT file',
            [('PDBQT files', '*.pdbqt'), ('All files', '*.*')])

        cx = ask_float('Center X', 'Grid center X:', 0.0)
        cy = ask_float('Center Y', 'Grid center Y:', 0.0)
        cz = ask_float('Center Z', 'Grid center Z:', 0.0)
        sx = ask_float('Size X', 'Box size X:', 30.0)
        sy = ask_float('Size Y', 'Box size Y:', 30.0)
        sz = ask_float('Size Z', 'Box size Z:', 30.0)

        # Flex distance is set once for the whole run in Cell 7 (applies to
        # every configured target), not per target -- don't ask for it here.
        if all(x is not None for x in [target_name, cx, cy, cz, sx, sy, sz]):
            flex_targets.append({
                'target_name': target_name.strip().lower().replace(' ', '_'),
                'pdb_id': pdb_id.strip().upper() if pdb_id else 'UNKNOWN',
                'receptor_pdbqt': receptor_pdbqt,
                'center': [cx, cy, cz],
                'size': [sx, sy, sz],
            })
            print(f'Added target: {target_name}')
            print(f'  Receptor: {receptor_pdbqt}')
            print(f'  Center: ({cx}, {cy}, {cz})')
            print(f'Total targets configured: {len(flex_targets)}')

def show_targets(b):
    with target_config_out:
        clear_output()
        if not flex_targets:
            print('No targets configured yet.')
            return
        for i, t in enumerate(flex_targets):
            print(f'Target {i+1}: {t["target_name"]} ({t["pdb_id"]})')
            print(f'  Receptor: {t["receptor_pdbqt"]}')
            print(f'  Center: {t["center"]}')
            print(f'  Size: {t["size"]}')
            print()

btn_add_target.on_click(add_target)
btn_show_targets.on_click(show_targets)

print('Add each target you want to dock against.')
print('Click Add Target for each one -- Candida, Aspergillus, Human.')
print('Flex distance is set once for the whole run in Cell 7, applied to all targets.')
display(btn_add_target)
display(btn_show_targets)
display(target_config_out)

### Cell 6 -- Define the flexible docking function

Uses GNINA with --flexdist_ligand to automatically identify flexible residues
from the initial rigid docking pose, then redocks at high exhaustiveness.

In [ ]:
def parse_gnina_affinities(pdbqt_path):
    """Extract affinity scores from GNINA output PDBQT."""
    affinities = []
    try:
        with open(pdbqt_path) as f:
            for line in f:
                if 'minimizedAffinity' in line:
                    try:
                        val = float(line.strip().split()[-1])
                        affinities.append(val)
                    except:
                        pass
    except:
        pass
    return sorted(affinities)


def extract_best_pose(pdbqt_path, output_path):
    """Keep only the first MODEL from a multi-pose PDBQT file."""
    lines = []
    in_first_model = False
    try:
        with open(pdbqt_path) as f:
            for line in f:
                if line.startswith('MODEL'):
                    if not in_first_model:
                        in_first_model = True
                        lines.append(line)
                    else:
                        break
                elif in_first_model:
                    lines.append(line)
        with open(output_path, 'w') as f:
            f.writelines(lines)
        return True
    except Exception as e:
        return False


def get_flexible_residues_from_output(flex_pdbqt_path):
    """Parse the flexible receptor output to get residue names."""
    residues = set()
    try:
        with open(flex_pdbqt_path) as f:
            for line in f:
                if line.startswith(('ATOM', 'HETATM')):
                    resname = line[17:20].strip()
                    resnum = line[22:26].strip()
                    chain = line[21].strip()
                    residues.add(f'{resname}{resnum}{chain}')
    except:
        pass
    return list(residues)


# AutoDock atom types for hydrogen -- 'H' is nonpolar, but polar/donor
# hydrogens are typed 'HD' and 'HS'. A bare `element != 'H'` check lets
# HD/HS hydrogens leak into the coordinate set used for contact distances.
HYDROGEN_TYPES = {'H', 'HD', 'HS'}


def parse_pdbqt_coords(pdbqt_path):
    """Extract heavy atom coordinates from first model."""
    coords = []
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    element = line[77:79].strip() if len(line) > 77 else 'X'
                    if element not in HYDROGEN_TYPES:
                        coords.append([x, y, z])
                except:
                    pass
            elif line.startswith('ENDMDL'):
                break
    return np.array(coords) if coords else np.array([])


def merge_receptor_with_flex(rigid_receptor_pdbqt, flex_pose_pdbqt, output_path):
    """Build the receptor structure as it actually exists in the docked pose.

    --out_flex only contains the residues GNINA set flexible, written in their
    final docked conformation -- the rest of the receptor stays in the input
    (rigid_receptor_pdbqt) file. Contacts must be measured against this merged
    structure, not the original rigid receptor, or every flexible residue's
    contact distance reflects its pre-docking position instead of its actual
    docked one.
    """
    flex_residue_keys = set()
    flex_lines = []
    with open(flex_pose_pdbqt) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                resname = line[17:20].strip()
                resnum = line[22:26].strip()
                chain = line[21].strip()
                flex_residue_keys.add((resname, resnum, chain))
                flex_lines.append(line)

    static_lines = []
    with open(rigid_receptor_pdbqt) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                resname = line[17:20].strip()
                resnum = line[22:26].strip()
                chain = line[21].strip()
                if (resname, resnum, chain) in flex_residue_keys:
                    continue
            static_lines.append(line)

    with open(output_path, 'w') as f:
        f.writelines(static_lines)
        f.writelines(flex_lines)
        f.write('END\n')


def find_contacts_pdbqt(pose_pdbqt, receptor_pdbqt, cutoff=4.5):
    """Find receptor residues within cutoff of docked ligand.

    `receptor_pdbqt` should be the merged structure from
    merge_receptor_with_flex(), not the original rigid receptor, so that
    flexible residues are measured at their actual docked positions.
    """
    from Bio.PDB import PDBParser
    ligand_coords = parse_pdbqt_coords(pose_pdbqt)
    if len(ligand_coords) == 0:
        return []
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('receptor', receptor_pdbqt)
    contacts = []
    seen = set()
    for model in structure:
        for chain in model:
            for residue in chain:
                resname = residue.get_resname()
                resnum = residue.id[1]
                chain_id = chain.id
                key = (resname, resnum, chain_id)
                if key in seen:
                    continue
                for atom in residue:
                    # PDBQT's AutoDock atom-type column doesn't line up with
                    # the standard PDB element column, so Bio.PDB's
                    # atom.element can't be trusted here -- use the atom name.
                    if atom.get_name().strip().upper().startswith('H'):
                        continue
                    diffs = ligand_coords - atom.get_coord()
                    min_dist = np.sqrt((diffs**2).sum(axis=1)).min()
                    if min_dist <= cutoff:
                        contacts.append((resname, resnum, chain_id, round(float(min_dist), 3)))
                        seen.add(key)
                        break
        break
    return sorted(contacts, key=lambda x: x[3])


def run_flexible_gnina(ligand_pdbqt, rigid_pose_pdbqt, receptor_pdbqt,
                       center, size, flexdist, output_path,
                       exhaustiveness=32, n_poses=3, device=0, timeout=3600):
    """
    Run GNINA with automatic flexible side chain selection.
    Uses --flexdist_ligand with the initial rigid pose to automatically
    identify binding site side chains to make flexible.

    Flexible docking with several flexible residues is far more expensive
    than rigid docking -- runtimes of 15+ minutes per compound-target pair
    are expected, so the subprocess timeout must give real runs enough room.
    """
    out_flex = str(Path(output_path).parent / (Path(output_path).stem + '_flex.pdbqt'))

    cmd = [
        GNINA_BIN,
        '--receptor', receptor_pdbqt,
        '--ligand', ligand_pdbqt,
        '--out', output_path,
        '--out_flex', out_flex,
        '--center_x', str(center[0]),
        '--center_y', str(center[1]),
        '--center_z', str(center[2]),
        '--size_x', str(size[0]),
        '--size_y', str(size[1]),
        '--size_z', str(size[2]),
        '--flexdist_ligand', rigid_pose_pdbqt,
        '--flexdist', str(flexdist),
        '--exhaustiveness', str(exhaustiveness),
        '--num_modes', str(n_poses),
        '--device', str(device),
        '--pose_sort_order', 'Energy',
        '--quiet'
    ]

    result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)

    if Path(output_path).exists() and Path(output_path).stat().st_size > 0:
        flex_residues = get_flexible_residues_from_output(out_flex) if Path(out_flex).exists() else []
        return True, result.stderr, flex_residues
    else:
        return False, result.stderr, []


print('Flexible docking functions defined.')
print(f'GNINA binary: {GNINA_BIN}')
print('Using --flexdist_ligand for automatic flexible residue selection.')

### Cell 7 -- Run flexible docking

Docks all selected hits against all configured targets with flexible side chains.
Uses exhaustiveness 32 (4x higher than rigid screen) for better sampling.
Expected runtime: 5-15 minutes per compound-target pair on GPU.

In [ ]:
import time

exhaustiveness_widget = widgets.IntSlider(
    value=32, min=8, max=64, step=8,
    description='Exhaustiveness:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
flexdist_override = widgets.FloatSlider(
    value=4.0, min=2.0, max=8.0, step=0.5,
    description='Flex distance (A):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

run_out = widgets.Output()
btn_run = widgets.Button(
    description='Run flexible docking',
    button_style='danger',
    layout=widgets.Layout(width='300px')
)

def run_flexible_docking(b):
    with run_out:
        clear_output()

        if not selected['db_path']:
            print('Select database first.')
            return
        if not flex_targets:
            print('Configure targets in Cell 5 first.')
            return
        if not selected.get('hit_data'):
            print('Run Cell 4 first to load hit data.')
            return

        output_dir = Path(selected['output_dir']) / 'flexible_poses'
        output_dir.mkdir(parents=True, exist_ok=True)

        conn = sqlite3.connect(selected['db_path'])
        cursor = conn.cursor()

        total_runs = len(selected['hit_data']) * len(flex_targets)
        completed = 0
        start_time = time.time()

        progress = widgets.IntProgress(
            value=0, min=0, max=total_runs,
            description='Flexible docking:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='700px')
        )
        status_label = widgets.Label(value='Starting...')
        display(progress)
        display(status_label)

        for target in flex_targets:
            target_name = target['target_name']
            target_dir = output_dir / target_name
            target_dir.mkdir(parents=True, exist_ok=True)

            for hit in selected['hit_data']:
                chembl_id = hit['chembl_id']
                name = hit['name']

                # Check if already done
                already = cursor.execute(
                    'SELECT COUNT(*) FROM flexible_docking_results WHERE chembl_id=? AND target_name=?',
                    (chembl_id, target_name)
                ).fetchone()[0]
                if already > 0:
                    print(f'Skipping {name} vs {target_name} -- already done.')
                    completed += 1
                    progress.value = completed
                    continue

                # Get rigid pose file
                pose_col = 'candida_pose' if 'candida' in target_name else \
                           'aspergillus_pose' if 'aspergillus' in target_name else 'human_pose'
                rigid_pose = hit.get(pose_col)
                rigid_affinity = hit.get(
                    'candida' if 'candida' in target_name else
                    'aspergillus' if 'aspergillus' in target_name else 'human'
                )

                if not rigid_pose or not Path(str(rigid_pose)).exists():
                    print(f'No rigid pose found for {name} vs {target_name} -- skipping.')
                    completed += 1
                    progress.value = completed
                    continue

                # Get ligand PDBQT -- find in database
                ligand_path = cursor.execute(
                    'SELECT pdbqt_path FROM molecules WHERE chembl_id=?',
                    (chembl_id,)
                ).fetchone()

                if not ligand_path or not ligand_path[0] or not Path(ligand_path[0]).exists():
                    # Try finding in fdapdbqt folder
                    fdapdbqt = Path.home() / 'BasementDrugDiscovery/data/fdapdbqt' / f'{chembl_id}.pdbqt'
                    if fdapdbqt.exists():
                        ligand_pdbqt = str(fdapdbqt)
                    else:
                        print(f'Ligand PDBQT not found for {chembl_id} -- skipping.')
                        completed += 1
                        progress.value = completed
                        continue
                else:
                    ligand_pdbqt = ligand_path[0]

                output_path = str(target_dir / f'{chembl_id}_flexible_pose.pdbqt')

                status_label.value = f'Running {name} vs {target_name} (exhaustiveness {exhaustiveness_widget.value})...'
                print(f'\nDocking {name} ({chembl_id}) vs {target_name}...')

                try:
                    success, stderr, flex_residues = run_flexible_gnina(
                        ligand_pdbqt=ligand_pdbqt,
                        rigid_pose_pdbqt=rigid_pose,
                        receptor_pdbqt=target['receptor_pdbqt'],
                        center=target['center'],
                        size=target['size'],
                        flexdist=flexdist_override.value,
                        output_path=output_path,
                        exhaustiveness=exhaustiveness_widget.value,
                        n_poses=3,
                        device=0
                    )

                    if success:
                        affinities = parse_gnina_affinities(output_path)
                        flex_affinity = affinities[0] if affinities else None
                        affinity_change = (flex_affinity - rigid_affinity) if \
                            (flex_affinity is not None and rigid_affinity is not None) else None

                        best_pose = output_path.replace('.pdbqt', '_best.pdbqt')
                        extract_best_pose(output_path, best_pose)

                        cursor.execute('''
                            INSERT OR IGNORE INTO flexible_docking_results
                            (chembl_id, target_name, pdb_id, rigid_affinity,
                             flexible_affinity, affinity_change, flexdist_used,
                             n_flexible_residues, flexible_residues,
                             exhaustiveness, pose_file, status, timestamp)
                            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
                        ''', (
                            chembl_id, target_name, target['pdb_id'],
                            rigid_affinity, flex_affinity, affinity_change,
                            flexdist_override.value, len(flex_residues),
                            ','.join(flex_residues),
                            exhaustiveness_widget.value, best_pose,
                            'success', datetime.now().isoformat()
                        ))

                        result_id = cursor.lastrowid

                        # Store contacts -- measured against the receptor as it
                        # actually exists in the docked pose (static parts +
                        # the flexible residues' docked positions from
                        # --out_flex), not the original rigid receptor.
                        out_flex = str(Path(output_path).parent / (Path(output_path).stem + '_flex.pdbqt'))
                        if Path(best_pose).exists() and Path(out_flex).exists():
                            best_flex = out_flex.replace('.pdbqt', '_best.pdbqt')
                            extract_best_pose(out_flex, best_flex)

                            merged_receptor = str(target_dir / f'{chembl_id}_merged_receptor.pdbqt')
                            merge_receptor_with_flex(target['receptor_pdbqt'], best_flex, merged_receptor)

                            contacts = find_contacts_pdbqt(best_pose, merged_receptor)
                            for resname, resnum, chain_id, dist in contacts:
                                is_flex = 1 if f'{resname}{resnum}{chain_id}' in flex_residues else 0
                                cursor.execute('''
                                    INSERT INTO flexible_docking_contacts
                                    (result_id, chembl_id, target_name,
                                     residue_name, residue_number, chain,
                                     contact_distance, is_flexible)
                                    VALUES (?,?,?,?,?,?,?,?)
                                ''', (result_id, chembl_id, target_name,
                                      resname, resnum, chain_id, dist, is_flex))
                        elif Path(best_pose).exists():
                            # No residues were set flexible (e.g. flexdist too
                            # tight) -- the rigid receptor is already accurate.
                            contacts = find_contacts_pdbqt(best_pose, target['receptor_pdbqt'])
                            for resname, resnum, chain_id, dist in contacts:
                                is_flex = 1 if f'{resname}{resnum}{chain_id}' in flex_residues else 0
                                cursor.execute('''
                                    INSERT INTO flexible_docking_contacts
                                    (result_id, chembl_id, target_name,
                                     residue_name, residue_number, chain,
                                     contact_distance, is_flexible)
                                    VALUES (?,?,?,?,?,?,?,?)
                                ''', (result_id, chembl_id, target_name,
                                      resname, resnum, chain_id, dist, is_flex))

                        conn.commit()
                        rigid_str = f'{rigid_affinity:.2f}' if rigid_affinity is not None else 'N/A'
                        flex_str = f'{flex_affinity:.2f}' if flex_affinity is not None else 'N/A'
                        change_str = f'{affinity_change:+.2f}' if affinity_change is not None else 'N/A'
                        print(f'  Rigid: {rigid_str} | Flexible: {flex_str} | Change: {change_str}')
                        print(f'  Flexible residues: {len(flex_residues)}')

                    else:
                        cursor.execute('''
                            INSERT OR IGNORE INTO flexible_docking_results
                            (chembl_id, target_name, pdb_id, rigid_affinity,
                             status, error_message, timestamp)
                            VALUES (?,?,?,?,?,?,?)
                        ''', (chembl_id, target_name, target['pdb_id'],
                              rigid_affinity, 'failed', stderr[:300],
                              datetime.now().isoformat()))
                        conn.commit()
                        print(f'  FAILED: {stderr[:100]}')

                except subprocess.TimeoutExpired:
                    # GNINA was still running when the timeout hit -- record
                    # this explicitly so it doesn't just vanish from the
                    # results table and get silently retried forever.
                    cursor.execute('''
                        INSERT OR IGNORE INTO flexible_docking_results
                        (chembl_id, target_name, pdb_id, rigid_affinity,
                         status, error_message, timestamp)
                        VALUES (?,?,?,?,?,?,?)
                    ''', (chembl_id, target_name, target['pdb_id'],
                          rigid_affinity, 'timeout',
                          'GNINA exceeded the subprocess timeout',
                          datetime.now().isoformat()))
                    conn.commit()
                    print(f'  TIMEOUT: GNINA did not finish in time.')
                except Exception as e:
                    print(f'  ERROR: {e}')

                completed += 1
                progress.value = completed
                elapsed = time.time() - start_time
                rate = completed / elapsed
                remaining = (total_runs - completed) / rate if rate > 0 else 0
                status_label.value = f'{completed}/{total_runs} -- Remaining: {remaining/60:.1f} min'

        conn.close()
        print(f'\nAll flexible docking runs complete.')


print('Set parameters and click Run:')
display(exhaustiveness_widget)
display(flexdist_override)
print('\nFlex distance: side chains within this distance of the rigid pose will be made flexible.')
print('4 Angstroms captures the immediate binding site. 6 Angstroms includes the outer shell.')
print('Note: flexible docking with several flexible residues can take well over 15 minutes per')
print('compound-target pair -- budget accordingly, especially at higher exhaustiveness.')
display(btn_run)
btn_run.on_click(run_flexible_docking)
display(run_out)

### Cell 8 -- Compare rigid vs flexible docking results

In [ ]:
if selected['db_path']:
    conn = sqlite3.connect(selected['db_path'])

    print('=== RIGID vs FLEXIBLE DOCKING COMPARISON ===')
    comparison = pd.read_sql('''
        SELECT
            fdr.chembl_id,
            m.name,
            fdr.target_name,
            fdr.rigid_affinity,
            fdr.flexible_affinity,
            fdr.affinity_change,
            fdr.n_flexible_residues,
            fdr.status
        FROM flexible_docking_results fdr
        JOIN molecules m ON fdr.chembl_id = m.chembl_id
        WHERE fdr.status = 'success'
        ORDER BY fdr.target_name, fdr.flexible_affinity ASC
    ''', conn)

    if len(comparison) > 0:
        display(comparison.to_string(index=False))

        # Plot comparison
        fig, axes = plt.subplots(1, len(comparison['target_name'].unique()),
                                  figsize=(6 * len(comparison['target_name'].unique()), 5))
        if len(comparison['target_name'].unique()) == 1:
            axes = [axes]

        for ax, (target, group) in zip(axes, comparison.groupby('target_name')):
            x = range(len(group))
            ax.bar([i - 0.2 for i in x], group['rigid_affinity'].abs(),
                   width=0.4, label='Rigid', color='steelblue', alpha=0.7)
            ax.bar([i + 0.2 for i in x], group['flexible_affinity'].abs(),
                   width=0.4, label='Flexible', color='coral', alpha=0.7)
            ax.set_xticks(list(x))
            ax.set_xticklabels(group['name'], rotation=20, ha='right', fontsize=9)
            ax.set_ylabel('|Affinity| kcal/mol', fontsize=11)
            ax.set_title(target, fontsize=10)
            ax.legend()
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

        plt.suptitle('Rigid vs Flexible Docking -- Top 5 Hits', fontsize=13, fontweight='bold')
        plt.tight_layout()

        fig_path = Path(selected['output_dir']) / 'rigid_vs_flexible_comparison.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f'\nFigure saved: {fig_path}')
        plt.show()
    else:
        print('No flexible docking results yet. Run Cell 7 first.')

    conn.close()
else:
    print('Select database first.')

### Cell 9 -- Contact residue comparison for top hit

Shows which residues Atogepant contacts in rigid vs flexible docking.
Highlights residues that changed conformation during flexible docking.

In [ ]:
contact_out = widgets.Output()
btn_contacts = widgets.Button(
    description='Show flexible docking contacts',
    button_style='info',
    layout=widgets.Layout(width='350px')
)

def show_flex_contacts(b):
    with contact_out:
        clear_output()
        chembl_id = ask_string('ChEMBL ID',
            'Enter ChEMBL ID to inspect:', 'CHEMBL3991065')
        target = ask_string('Target',
            'Enter target name:', 'candida_albicanscyp51')

        if not chembl_id or not target:
            return

        conn = sqlite3.connect(selected['db_path'])

        result = pd.read_sql('''
            SELECT fdr.chembl_id, m.name, fdr.target_name,
                   fdr.rigid_affinity, fdr.flexible_affinity,
                   fdr.affinity_change, fdr.n_flexible_residues,
                   fdr.flexible_residues
            FROM flexible_docking_results fdr
            JOIN molecules m ON fdr.chembl_id = m.chembl_id
            WHERE fdr.chembl_id = ? AND fdr.target_name = ?
        ''', conn, params=(chembl_id, target))

        if len(result) == 0:
            print(f'No flexible docking result for {chembl_id} vs {target}')
            conn.close()
            return

        row = result.iloc[0]
        print(f'Compound: {row["name"]} ({chembl_id})')
        print(f'Target: {target}')
        print(f'Rigid affinity: {row["rigid_affinity"]:.2f} kcal/mol')
        print(f'Flexible affinity: {row["flexible_affinity"]:.2f} kcal/mol')
        print(f'Affinity change: {row["affinity_change"]:+.2f} kcal/mol')
        print(f'Number of flexible residues: {row["n_flexible_residues"]}')
        print(f'Flexible residues: {row["flexible_residues"]}')

        contacts = pd.read_sql('''
            SELECT residue_name, residue_number, chain,
                   contact_distance, is_flexible
            FROM flexible_docking_contacts
            WHERE chembl_id = ? AND target_name = ?
            ORDER BY contact_distance ASC
        ''', conn, params=(chembl_id, target))

        conn.close()

        if len(contacts) > 0:
            print(f'\nContacting residues (is_flexible=1 means side chain moved during docking):')
            display(contacts)

            heme = contacts[contacts['residue_name'] == 'HEM']
            if len(heme) > 0:
                print(f'\nHeme contact confirmed at {heme.iloc[0]["contact_distance"]:.3f} Angstroms')
            else:
                print('\nNo heme contact in flexible docking pose.')

btn_contacts.on_click(show_flex_contacts)
display(btn_contacts)
display(contact_out)

---
## Interpreting the Results

**Affinity change after flexible docking:**
- More negative change (e.g. -1.5 kcal/mol) means the compound binds better when the protein can adjust. This suggests the rigid receptor was penalizing the compound for steric clashes that the protein would normally accommodate.
- Small change (less than 0.5 kcal/mol) means the rigid docking was already finding a good pose -- the binding site geometry fits well without protein movement.
- Positive change means the compound scored worse with flexibility, which can indicate the rigid pose was artificially favorable.

**For the FDA submission:**
Flexible docking scores that remain strong or improve relative to rigid docking increase confidence in the prediction. Compounds that maintain heme contact with flexible side chains are the most credible hits.

**GitHub:** https://github.com/sardism/BasementDrugDiscovery

**Note:** All results are computational predictions requiring experimental validation.